In [ ]:
import os
import time
import pandas as pd
import numpy as np
from shapely import wkt
from geopy.distance import geodesic
from shapely.geometry import Point
from shapely.errors import WKTReadingError

In [ ]:
# Get current script path
current_path = os.path.dirname(os.path.abspath(__file__))

# File paths
link_file = os.path.join(current_path, "lts_link_gmns.csv")
node_file = os.path.join(current_path, "lts_node_gmns.csv")
node_taz_file = os.path.join(current_path, "TAZ_centroid.csv")

# Default connector configuration
DEFAULT_CONNECTOR_CONFIG = {
    "dir_flag": 1,
    "lanes": 1,
    "free_speed": 5,
    "capacity": 10000,
    "link_type_name": "connector",
    "link_type": 0,
    "allowed_uses": "walk",
    "from_biway": 1,
    "is_link": 0,
    "vdf_toll": 0,
    "vdf_alpha": 0.15,
    "vdf_beta": 4,
    "vdf_plf": 1,
}

In [ ]:
# Load CSVs
link_df = pd.read_csv(link_file)
node_df = pd.read_csv(node_file)
node_taz_df = pd.read_csv(node_taz_file)

# Start timing
start_time = time.time()

In [ ]:
# %%
def process_node_data(node_df, node_taz_df, output_path=None):
    try:
        print("Starting to process node data...")

        # Step 1: Find the maximum node_id in node_taz_df
        print("Finding the maximum node_id in node_taz_df...")
        max_node_id_taz = node_taz_df['zone_id'].max()
        min_node_id = node_df['node_id'].min()
        print(f"Maximum node_id in node_taz_df: {max_node_id_taz}")

        # Step 2: Add (max_node_id_taz + 1) to all node_ids in node_df
        print("Adding new_node_id to node_df...")
        node_df['new_node_id'] = node_df['node_id'] + max_node_id_taz - min_node_id + 1
        print("New node_id generation completed.")
        return node_df

    except Exception as e:
        print(f"An error occurred while processing node data: {e}")


# %%
def generate_connector_links(updated_node_df, node_taz_df, output_path=None, n_connector=3, node_types=None,
    connector_config=None):
    try:
        print("Starting to generate connector links...")

        # Set default configuration
        if connector_config is None:
            connector_config = DEFAULT_CONNECTOR_CONFIG.copy()

        # Force x/y coordinate columns to be float
        updated_node_df[['x_coord', 'y_coord']] = updated_node_df[['x_coord', 'y_coord']].astype(float)
        node_taz_df[['x_coord', 'y_coord']] = node_taz_df[['x_coord', 'y_coord']].astype(float)
        
        connector_links = []
        total_length = 0
        pair_number = 0
        print(updated_node_df['node_type'].unique())

        # Filter node type only when specified (Modified May 12, 2026)
        if node_types is not None:
            updated_node_df = updated_node_df[
            updated_node_df['node_type'].isin(node_types)]

        for idx, taz_node in node_taz_df.iterrows():
            taz_node_id = taz_node['zone_id']
            taz_node_x = taz_node['x_coord']
            taz_node_y = taz_node['y_coord']

            updated_node_df['distance'] = np.sqrt(
                (updated_node_df['x_coord'] - taz_node_x) ** 2 +
                (updated_node_df['y_coord'] - taz_node_y) ** 2
            )

            # nearest # nodes
            nearest_nodes = updated_node_df.nsmallest(n_connector, 'distance')
            for _, nearest_node in nearest_nodes.iterrows():

                nearest_node_id = nearest_node['new_node_id']
                nearest_node_x = nearest_node['x_coord']
                nearest_node_y = nearest_node['y_coord']

                for from_id, to_id, from_x, from_y, to_x, to_y in [
                    (nearest_node_id, taz_node_id, nearest_node_x, nearest_node_y, taz_node_x, taz_node_y),
                    (taz_node_id, nearest_node_id, taz_node_x, taz_node_y, nearest_node_x, nearest_node_y)
                ]:

                    geometry = f"LINESTRING ({from_x} {from_y}, {to_x} {to_y})"

                    length = geodesic(
                        (from_y, from_x),
                        (to_y, to_x)
                    ).meters

                    connector_links.append({
                        "link_id": len(connector_links) + 1,
                        "from_node_id": from_id,
                        "to_node_id": to_id,
                        "length": length,
                        "geometry": geometry,
                        **connector_config
                    })

        connector_links_df = pd.DataFrame(connector_links)
        print(f"Generated {len(connector_links_df)} connector links.")

        connector_links_df["vdf_toll"] = (connector_config["vdf_toll"])
        connector_links_df["allowed_uses"] = (connector_config["allowed_uses"])
        connector_links_df["vdf_alpha"] = (connector_config["vdf_alpha"])
        connector_links_df["vdf_beta"] = (connector_config["vdf_beta"])
        connector_links_df["vdf_plf"] = (connector_config["vdf_plf"])
        connector_links_df["vdf_length_mi"] = connector_links_df["length"] / 1609
        connector_links_df["vdf_free_speed_mph"] = connector_links_df["free_speed"] / 1.60934
        connector_links_df["free_speed_in_mph_raw"] = round(connector_links_df["vdf_free_speed_mph"] / 5) * 5
        connector_links_df["vdf_fftt"] = (connector_links_df["length"] / connector_links_df["free_speed"]) * 0.06

        other_columns = ['ref_volume', 'base_volume', 'base_vol_auto', 'restricted_turn_nodes']
        for other_column in other_columns:
            connector_links_df[other_column] = None

        file_name = "connector_links.csv"
        output_file = os.path.join(output_path, file_name)
        if output_file:
            connector_links_df.to_csv(output_file, index=False)
            print(f"The connector links have been successfully saved to '{output_file}'.")
        else:
            print("Output file not provided. Skipping file saving.")

        return connector_links_df

    except Exception as e:
        print(f"An error occurred while generating connector links: {e}")


# %%
def update_and_merge_links(link_df, updated_node_df, connector_links_df, output_path):
    """
    Updates link_df with new_node_id, merges it with connector_links_df, and saves the updated file.

    Args:
        link_df (pd.DataFrame): DataFrame containing the original link data.
        node_df (pd.DataFrame): DataFrame containing node_id and new_node_id mapping.
        connector_links_df (pd.DataFrame): DataFrame containing the connector links.
        output_file (str): Path to save the updated Link_Updated.csv file.
    """
    try:
        # Step 1: Create a mapping of node_id to new_node_id
        node_id_map = dict(zip(updated_node_df['node_id'], updated_node_df['new_node_id']))

        # Step 2: Update from_node_id and to_node_id in link_df
        link_df['from_node_id'] = link_df['from_node_id'].map(node_id_map)
        link_df['to_node_id'] = link_df['to_node_id'].map(node_id_map)

        # Step 3: Validate if there are any unmatched IDs
        if link_df['from_node_id'].isnull().any() or link_df['to_node_id'].isnull().any():
            print("Warning: Some from_node_id or to_node_id in link_df could not be mapped to new_node_id.")

        # Step 3.5: Add new column to link_df
        # Step3.5 Add new columns
        link_df["vdf_toll"] = 0
        link_df["allowed_uses"] = 'bike'
        link_df["vdf_alpha"] = 0.15
        link_df["vdf_beta"] = 4
        link_df["vdf_plf"] = 1
        link_df["vdf_length_mi"] = link_df["length"] / 1609
        link_df["vdf_free_speed_mph"] = link_df["free_speed"] / 1.60934
        link_df["free_speed_in_mph_raw"] = round(link_df["vdf_free_speed_mph"] / 5) * 5
        link_df["vdf_fftt"] = (link_df["length"] / link_df["free_speed"]) * 0.06

        other_columns = ['ref_volume', 'base_volume', 'base_vol_auto', 'restricted_turn_nodes']
        for other_column in other_columns:
            link_df[other_column] = None

        # Step 4: Align columns between link_df and connector_links_df
        all_columns = set(link_df.columns).union(connector_links_df.columns)

        # Add missing columns with None
        for col in all_columns:
            if col not in link_df.columns:
                link_df[col] = None
            if col not in connector_links_df.columns:
                connector_links_df[col] = None

        # Ensure connector_links_df has the same column order as link_df
        connector_links_df = connector_links_df[link_df.columns]
        # breakpoint()

        # Step 5: Combine link_df and connector_links_df
        combined_links_df = pd.concat([link_df, connector_links_df], ignore_index=True)

        # Step 6: Sort and assign new link_id
        combined_links_df = combined_links_df.sort_values(by=['from_node_id', 'to_node_id']).reset_index(drop=True)
        combined_links_df['link_id'] = range(1, len(combined_links_df) + 1)

        # Step 7: Save the updated DataFrame to the output file
        file_name = "link_updated.csv"
        output_file = os.path.join(output_path, file_name)
        combined_links_df.to_csv(output_file, index=False)
        print(f"Updated and merged data has been saved to {output_file}.")

    except Exception as e:
        print(f"An error occurred: {e}")


# %%
def create_updated_node_df(updated_node_df, node_taz_df, output_path):
    try:
        updated_node_df = updated_node_df.rename(columns={'node_id': 'old_node_id'})
        updated_node_df = updated_node_df.rename(columns={'new_node_id': 'node_id'})

        updated_node_df['zone_id'] = None
        node_taz_df['node_id'] = node_taz_df['zone_id']

        Node_Updated_df = pd.concat([node_taz_df, updated_node_df], ignore_index=True)
        Node_Updated_df = Node_Updated_df.sort_values(by=['node_id']).reset_index(drop=True)
        Node_Updated_df = Node_Updated_df.drop(columns=['ctrl_type', 'distance'])


        '''
        for i in range(len(Node_Updated_df)):
            if pd.isna(Node_Updated_df.loc[i, 'geometry']) or Node_Updated_df.loc[i, 'geometry'].strip() == '':
                x_coord = Node_Updated_df.loc[i, 'x_coord']
                y_coord = Node_Updated_df.loc[i, 'y_coord']
                Node_Updated_df.loc[i, 'geometry'] = f"POINT ({x_coord} {y_coord})"
        '''
        # 1. Extract rows where 'geometry' is NaN or empty string
        missing_geometry_df = Node_Updated_df[
        Node_Updated_df['geometry'].isna() | (Node_Updated_df['geometry'].str.strip() == '')
        ].copy()

        # 2. Remove those rows from the original DataFrame
        Node_Updated_df = Node_Updated_df.drop(missing_geometry_df.index)

        # 3. Fill 'geometry' column using 'x_coord' and 'y_coord'
        missing_geometry_df['geometry'] = missing_geometry_df.apply(
            lambda row: f"POINT ({row['x_coord']} {row['y_coord']})", axis=1
        )

        # 4. Concatenate the two DataFrames and sort by original index to preserve row order
        Node_Updated_df = pd.concat([Node_Updated_df, missing_geometry_df])
        Node_Updated_df = Node_Updated_df.sort_index()

        # 5. Delete the temporary DataFrame
        del missing_geometry_df

        file_name = "node_updated.csv"
        output_file = os.path.join(output_path, file_name)
        #Node_Updated_df = Node_Updated_df.drop(columns='zone_id', errors='ignore').rename(columns={'zone_id': 'zone_id'})
        
        #Important!! MOVE node_id to the 1st column
        Node_Updated_df = Node_Updated_df[['node_id'] + [c for c in Node_Updated_df.columns if c != 'node_id']]
        Node_Updated_df.to_csv(output_file, index=False)       
        print(f"The updated node data has been successfully saved to '{output_file}'.")
        return Node_Updated_df

    except Exception as e:
        print(f"An error occurred: {e}")


In [ ]:
# Example usage
output_path = current_path
updated_node_df = process_node_data(node_df, node_taz_df, output_path)
connector_links_df = generate_connector_links(updated_node_df, node_taz_df, output_path, n_connector=3, node_types=None,
    connector_config=None)
update_and_merge_links(link_df, updated_node_df, connector_links_df, output_path)
Node_Updated_df = create_updated_node_df(updated_node_df, node_taz_df, output_path)


In [ ]:
# End timing
end_time = time.time()
print(f"Computational time: {end_time - start_time:.2f} seconds")